# ERP → Google Sheets Daily Automation

This notebook generates report DataFrames, builds a local `master_df`, creates a strict JSON-safe payload, and uploads one previous-day batch to Google Apps Script.

In [23]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    APPS_SCRIPT_URL,
    API_TOKEN,
    SPREADSHEET_ID,
    REPORT_TIMEZONE,
    get_previous_report_date,
    format_sheet_name,
    validate_config,
)
from src.master_builder import build_master_dataframe
from src.json_payload import build_payload
from src.google_uploader import (
    check_apps_script_health,
    upload_to_google_apps_script,
)

validate_config()

print("Project root    :", PROJECT_ROOT)
print("Report timezone :", REPORT_TIMEZONE)
print("Spreadsheet ID  :", SPREADSHEET_ID)
print("Apps Script URL :", APPS_SCRIPT_URL)


Project root    : c:\Users\Sujon\Desktop\DoIT\ERP_GoogleSheet_Automation_Hardcoded
Report timezone : Asia/Dhaka
Spreadsheet ID  : 1944VCMThyqliN_VNRyJggHEf6j3cVMUw-d__8j9i6I0
Apps Script URL : https://script.google.com/macros/s/AKfycbzT3Di2kguDbdfPaongnvyBuw0cp6rjcNJ-Z7pvyW7bG2ZhTAziF32u0IJg_WdOT09K/exec


## 1. Optional Apps Script health check

Run this during setup/troubleshooting. It does not write a daily report.

In [24]:
health = check_apps_script_health()
print(health)
if not health.get("success"):
    raise RuntimeError(health.get("error", "Apps Script health check failed"))


{'success': True, 'message': 'ERP Google Sheets Web App is running'}


## 2. Generate your real ERP reports

Replace the sample DataFrames below with your existing ERP report-generation code. Do not change the upload architecture.

## Data Process Start Here

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from bs4 import BeautifulSoup

### Less Production Report

In [26]:
import pandas as pd

# ---------------------------------------------------------
# 1. Get Previous Day
# ---------------------------------------------------------
previous_day = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)

# ---------------------------------------------------------
# 2. Format Previous Day as DDMMYYYY
# ---------------------------------------------------------
date_str = previous_day.strftime("%d%m%Y")

In [27]:
date_str

'23092026'

In [28]:
# ---------------------------------------------------------
# 3. Build Dynamic File Path
# ---------------------------------------------------------
file_path = fr"..\Data\SDEA_{date_str}.xlsx"

# ---------------------------------------------------------
# 4. Load Dataset
# ---------------------------------------------------------
sdea_df = pd.read_excel(file_path)

# Preview
sdea_df.head(5)

,SL#,Line,Buyer,Style,Work Day,SMV,IE Target 100%,Day Target Qty,Achieve Qty,Achieve (%),Today Efficiency (%),Monthly Efficiency (%),Previous Day Efficiency (%),Today Produce Minute,Today Availabe Minute,Monthly Produce Minute,Monthly Availabe Minute,Factory
0,1,Line 01,NEXT,Y63406 - GIRLS L/S T-SHIRT,4,5.80,375,375,370,98.67,38.60,45.10,46.24,15750.0,40800.0,363870.55,805425.0,MGL
1,2,Line 01,ECI,S27823HCHAMB - MENS L/S SHIRT,1,17.90,755,755,760,100.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MGL
2,3,Line 02,ECI,SS27224-86P - LADIES L/S SHIRT,1,18.50,711,711,540,75.95,44.52,46.70,62.32,14800.0,33240.0,328496.51,702090.0,MGL
3,4,Line 02,ECI,SS27 224-86S - LADIES L/S SHIRT,1,18.50,264,264,260,98.48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MGL
4,5,Line 03,ECI,S27105HBLANCO - LADIES L/S SHIRT,2,23.45,594,594,480,80.81,36.50,40.57,37.22,11256.0,30840.0,282834.50,691515.0,MGL


In [29]:
# ---------------------------------------------------------
# 1. Get Previous Day
# ---------------------------------------------------------
previous_day = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)

# Copy source DataFrame
report_df = sdea_df.copy()


# ---------------------------------------------------------
# 2. Filter Previous Day Data
# ---------------------------------------------------------
# Find a suitable date column
date_columns = [
    col
    for col in report_df.columns
    if any(
        word in str(col).lower()
        for word in ["date", "work date", "production date"]
    )
]

# If a date column exists, filter for previous day
if date_columns:
    date_col = date_columns[0]

    report_df[date_col] = pd.to_datetime(
        report_df[date_col],
        errors="coerce"
    )

    report_df = report_df[
        report_df[date_col].dt.normalize() == previous_day
    ].copy()


# ---------------------------------------------------------
# 3. Remove Invalid / Total Lines
# ---------------------------------------------------------
report_df = report_df[
    report_df["Factory"].notna() &
    report_df["Line"].notna() &
    (
        report_df["Line"]
        .astype(str)
        .str.strip()
        .str.lower() != "total"
    )
].copy()


# ---------------------------------------------------------
# 4. Convert Required Columns to Numeric
# ---------------------------------------------------------
report_df["Day Target Qty"] = pd.to_numeric(
    report_df["Day Target Qty"],
    errors="coerce"
)

report_df["Achieve Qty"] = pd.to_numeric(
    report_df["Achieve Qty"],
    errors="coerce"
)

report_df["Achieve (%)"] = pd.to_numeric(
    report_df["Achieve (%)"],
    errors="coerce"
)


# ---------------------------------------------------------
# 5. Filter Achieve (%) <= 70
# ---------------------------------------------------------
# Use the existing Achieve (%) value directly.
# No calculation or grouping is performed.
less_production = report_df[
    report_df["Achieve (%)"] <= 70
].copy()


# ---------------------------------------------------------
# 6. Add "Less Production" Column
# ---------------------------------------------------------
less_production.insert(
    0,
    "Less Production",
    ["Yesterday"] + [""] * (len(less_production) - 1)
)


# ---------------------------------------------------------
# 7. Rename Percentage Column
# ---------------------------------------------------------
less_production = less_production.rename(
    columns={
        "Achieve (%)": "Percentage"
    }
)


# ---------------------------------------------------------
# 8. Select Required Columns
# ---------------------------------------------------------
less_production = less_production[
    [
        "Less Production",
        "Factory",
        "Line",
        "Buyer",
        "Style",
        "Day Target Qty",
        "Achieve Qty",
        "Percentage"
    ]
].copy()


# ---------------------------------------------------------
# 9. Sort Data
# ---------------------------------------------------------
less_production = less_production.sort_values(
    ["Factory", "Line"]
).reset_index(drop=True)


# ---------------------------------------------------------
# 10. Format Percentage
# ---------------------------------------------------------
less_production["Percentage"] = (
    less_production["Percentage"]
    .round(2)
    .map(lambda x: f"{x:.2f}")
)


# ---------------------------------------------------------
# 11. Format Quantity Columns
# ---------------------------------------------------------
less_production["Day Target Qty"] = (
    less_production["Day Target Qty"]
    .apply(lambda x: f"{x:,.0f}" if pd.notna(x) else "")
)

less_production["Achieve Qty"] = (
    less_production["Achieve Qty"]
    .apply(lambda x: f"{x:,.0f}" if pd.notna(x) else "")
)


# ---------------------------------------------------------
# 12. Display Report
# ---------------------------------------------------------
display(
        f"Less Production — {previous_day:%d-%b-%Y}"
)

display(less_production)

'Less Production — 23-Sep-2026'

,Less Production,Factory,Line,Buyer,Style,Day Target Qty,Achieve Qty,Percentage
0,,MGL,Line 01,STREETONE,346497/3413355,582,400,68.73
1,,MGL,Line 02,US POLO,AW26-FM-SKT-002,364,120,32.97
2,Yesterday,MGL,Line 06,NEXT,Y63406 - GIRLS PANT,"1,500","1,000",66.67
3,,MGL,Line 07,ECI,S27105HRAYAF - LADIES LONG PANT,780,500,64.10
4,,MGL,Line 08,H&M,EQ LUKE COLLARLESS SHIRT,980,410,41.84
5,,MGL,Line 09,ECI,S27823HCHAMB - MENS L/S SHIRT,0,0,0.00
6,,MGNSL,Line 06,TEDDY S.P.A.,ABTD67420PFIO,240,160,66.67
7,,MGNSL,Line 13,TEDDY S.P.A.,ABTD67420PFIO,184,113,61.41


### Fabric MRR Pending Report

In [30]:
# ---------------------------------------------------------
# 3. Build Dynamic File Path
# ---------------------------------------------------------
file_path = fr"..\Data\MRR_Pending_Fabric_{date_str}.xlsx"

# ---------------------------------------------------------
# 4. Load Dataset
# ---------------------------------------------------------
fabric_mrr_pending_df = pd.read_excel(file_path)

# Preview
fabric_mrr_pending_df.head(5)


,SL#,Booking No,Supplier,Style No,Season,TOD Start,TOD End,Buyer,Fabric,PI NO,PI Date,PI Qty,BB LC No,Receive Qty,Merchant,Left Qty
0,1,MGNSL-FPO-26F0157,"CHANGSHU HJC CO.,LTD.",MFM_FW27_BER_0002,NaN,2026-11-20,2026-12-05,LOJAS RENNER,54% LYOCELL 46% COTTON (BCI) TWILL SOLID DYED ...,2026-ITS-HJC031,2026-09-15,4550.0,107926060251,0.0,Nipun,4550.0
1,2,MGNSL-FPO-26F0157,"CHANGSHU HJC CO.,LTD.",MFM_SS27_CAL_0058,NaN,2026-12-05,2026-12-05,LOJAS RENNER,54% LYOCELL 46% COTTON (BCI) TWILL SOLID DYED ...,2026-ITS-HJC031,2026-09-15,5950.0,107926060251,0.0,Nipun,5950.0
2,3,MGNSL-FPO-26F0154,"NANTONG SHUANGYING TEXTILE TECHNOLOGY CO.,LTD",MFM_SS26_MAC_0019B,FW27,2026-11-15,2026-11-15,LOJAS RENNER,55% LILEN 45% COTTON (BCI) Y/D SLUB CHAMBRAY W...,2026-ITS-NST008-C,2026-09-22,2520.0,107926060247,0.0,Nipun,2520.0
3,4,MGNSL-FPO-26L0091,Silk International Ltd.,R75002,SS27,2026-11-18,2026-12-15,NITEX PTE LTD.,"100% BCI COTTON, 154X114/60X60, 154x114/60x60",SIL/MGL/051/2026,2026-09-15,1599.0,1079260401062,0.0,Babu,1599.0
4,5,MGNSL-FPO-26L0091,Silk International Ltd.,R75002,SS27,2026-11-18,2026-12-15,NITEX PTE LTD.,95% cotton 5% spnadx 1x1 rib 280gsm,SIL/MGL/051/2026,2026-09-15,114.0,1079260401062,0.0,Babu,114.0


In [31]:
fabric_mrr_pending_df.columns.tolist()


['SL#',
 'Booking No',
 'Supplier',
 'Style No',
 'Season',
 'TOD Start',
 'TOD End',
 'Buyer',
 'Fabric',
 'PI NO',
 'PI Date',
 'PI Qty',
 'BB LC No',
 'Receive Qty',
 'Merchant',
 'Left Qty']

In [32]:
import pandas as pd
from datetime import date


# ---------------------------------------------------------
# 1. Normalize Column Names
# ---------------------------------------------------------
normalized_columns = {
    str(column).strip().casefold(): column
    for column in fabric_mrr_pending_df.columns
}


# ---------------------------------------------------------
# 2. Resolve Columns Dynamically
# ---------------------------------------------------------
def resolve_column(candidates, required=True):
    column = next(
        (
            normalized_columns[name.strip().casefold()]
            for name in candidates
            if name.strip().casefold() in normalized_columns
        ),
        None,
    )

    if required and column is None:
        available = ", ".join(str(name) for name in fabric_mrr_pending_df.columns)

        raise KeyError(
            f"Could not find a required column. "
            f"Tried: {', '.join(candidates)}. "
            f"Available columns: {available}"
        )

    return column


# ---------------------------------------------------------
# 3. Resolve Required / Optional Columns
# ---------------------------------------------------------
date_col = resolve_column([
    "Last TOD Date",
    "TOD End",
    "TOD Start"
])

style_col = resolve_column([
    "Style",
    "Style No",
    "Style No."
])

qty_col = resolve_column([
    "Qty",
    "Left Qty",
    "Less Qty",
    "Pending Qty",
    "Balance Qty"
])

buyer_col = resolve_column(
    ["Buyer"],
    required=False
)

merchant_col = resolve_column(
    ["Merchant"],
    required=False
)


# ---------------------------------------------------------
# 4. Prepare Working DataFrame
# ---------------------------------------------------------
today = pd.Timestamp(date.today())
end_date = today + pd.Timedelta(days=32)

work = fabric_mrr_pending_df.copy()

work["TOD"] = (
    pd.to_datetime(
        work[date_col],
        errors="coerce"
    )
    .dt.normalize()
)

work["Qty"] = (
    pd.to_numeric(
        work[qty_col],
        errors="coerce"
    )
    .fillna(0)
)

work["Buyer"] = (
    work[buyer_col]
    if buyer_col
    else ""
)

work["Merchant"] = (
    work[merchant_col]
    if merchant_col
    else ""
)


# ---------------------------------------------------------
# 5. Filter Today to Today + 32 Days
# ---------------------------------------------------------
work = work.loc[
    work["TOD"].between(
        today,
        end_date
    )
].copy()


# ---------------------------------------------------------
# 6. Create Days Category
# ---------------------------------------------------------
days_from_today = (
    work["TOD"] - today
).dt.days

work["Days"] = pd.cut(
    days_from_today,
    bins=[-1, 10, 20, 32],
    labels=[
        "10 Days",
        "20 Days",
        "30 Days"
    ]
)


# ---------------------------------------------------------
# 7. Combine Unique Names
# ---------------------------------------------------------
def unique_names(values):
    names = (
        pd.Series(values)
        .dropna()
        .astype(str)
        .str.strip()
    )

    names = names[names.ne("")]

    return ", ".join(
        pd.unique(names)
    )


# ---------------------------------------------------------
# 8. Create Main Report
# ---------------------------------------------------------
report = (
    work
    .groupby(
        [style_col, "TOD"],
        dropna=False,
        as_index=False
    )
    .agg(
        Buyer=("Buyer", unique_names),
        Qty=("Qty", "sum"),
        Merchant=("Merchant", unique_names),
        Days=("Days", "first"),
    )
    .rename(
        columns={
            style_col: "Style"
        }
    )
    [
        [
            "Days",
            "TOD",
            "Buyer",
            "Style",
            "Qty",
            "Merchant"
        ]
    ]
    .sort_values(
        ["TOD", "Style"]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 9. Add "Fabric Delay" as First Column
# ---------------------------------------------------------
report.insert(
    0,
    "Fabric Delay",
    ""
)

# Put "Yesterday" only in the first row
if not report.empty:
    report.loc[report.index[0], "Fabric Delay"] = "Yesterday"


# ---------------------------------------------------------
# 10. Final DataFrame
# ---------------------------------------------------------
fabric_delay_df = report


# ---------------------------------------------------------
# 11. Return / Display
# ---------------------------------------------------------
fabric_delay_df

,Fabric Delay,Days,TOD,Buyer,Style,Qty,Merchant
0,Yesterday,10 Days,2026-09-28,H&M,1257452,31696.00,Fakrul
1,,10 Days,2026-09-28,NITEX PTE LTD.,AE2337,19251.00,Babu
2,,10 Days,2026-09-30,NITEX PTE LTD.,AE2401,21300.00,Babu
3,,10 Days,2026-09-30,Fashion Fleet Clothing,Woman Top Bottom Set (Shirt+TrauserPaijama)(Set),11306.30,Sabbir
4,,10 Days,2026-10-01,TMS Fashion (HK) Ltd.,LISA STRETCH - SOLID,4214.79,Nipun
5,,10 Days,2026-10-01,TMS Fashion (HK) Ltd.,LISA STRETCH - STRIPE,1639.98,Nipun
6,,10 Days,2026-10-01,TMS Fashion (HK) Ltd.,MEDINA - SIGNAIPREIS STRIPE,7336.63,Nipun
7,,10 Days,2026-10-01,TMS Fashion (HK) Ltd.,Medina - Signalpreis uni,5379.00,Nipun
8,,10 Days,2026-10-01,TMS Fashion (HK) Ltd.,VERA NEW - NOS,2760.00,Nipun
9,,10 Days,2026-10-01,TMS Fashion (HK) Ltd.,Vera new - solid,552.00,Nipun


### Line Wise Earning Report

In [33]:
import pandas as pd

# ---------------------------------------------------------
# 3. Build Dynamic File Path
# ---------------------------------------------------------
file_path = fr"..\Data\DailySewingInfo_{date_str}.xlsx"

# ---------------------------------------------------------
# 4. Load Dataset
# ---------------------------------------------------------
DailySewingInfo = pd.read_excel(file_path)

# Preview
DailySewingInfo.head(5)

,Factory Name,Line No,Prod. Date,Working Hour,Buyer,Style Name,Style Code,SMV,FOB,CM,Order No,Body Part,Order Qty,Raw Qty,Target,Sewing Qty,Total CM,First TOD,Last TOD
0,MGL,Line-1,2026-09-23,NaN,EL CORTE INGLES,S27823HCHAMB,S27823HCHAMB,20.0,6.5000,1.9,0110990351301,NaN,9000,9000,NaN,760,1444.0,2026-10-15,2026-10-15
1,MGL,Line-1,2026-09-23,10.0,Geniemode Bd,Y63406,Y63406,22.0,6.8425,1.2,CJ8474434,NaN,7364,14728,NaN,370,444.0,2026-07-10,2026-07-10
2,MGL,Line-2,2026-09-23,NaN,EL CORTE INGLES,SS27 224-86P,SS27 224-86P,20.0,6.0000,2.1,0141591299001,NaN,450,450,NaN,40,84.0,2026-10-30,2026-10-30
3,MGL,Line-2,2026-09-23,NaN,EL CORTE INGLES,SS27 224-86P,SS27 224-86P,20.0,6.0000,2.1,0141591299001,NaN,450,450,NaN,500,1050.0,2026-10-30,2026-10-30
4,MGL,Line-2,2026-09-23,8.0,EL CORTE INGLES,SS27 224-86S,SS27 224-86S,20.0,6.0000,2.1,0141591299001,NaN,900,900,NaN,260,546.0,2026-10-30,2026-10-30


In [34]:
import pandas as pd


# ============================================================
# 1. SETTINGS
# ============================================================


FIXED_LINE_COST = 1925

FACTORIES = [
    "MGNSL",
    "MGL",
    "MGSL",
    "MGSL-DK",
]

LINE_WISE_TEXT = "*Quantity * CM - Fixed Line Cost Per Day USD $ 1925"


# ============================================================
# 3. CLEAN COLUMN NAMES
# ============================================================

DailySewingInfo.columns = (
    DailySewingInfo.columns
      .astype(str)
      .str.strip()
)


# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Factory Name",
    "Line No",
    "CM",
    "Sewing Qty"
]

missing_columns = [
    col for col in required_columns
    if col not in DailySewingInfo.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )


# ============================================================
# 5. CLEAN REQUIRED DATA
# ============================================================

DailySewingInfo["Factory Name"] = (
    DailySewingInfo["Factory Name"]
    .astype(str)
    .str.strip()
)

DailySewingInfo["Line No"] = (
    DailySewingInfo["Line No"]
    .astype(str)
    .str.strip()
)

DailySewingInfo["CM"] = pd.to_numeric(
    DailySewingInfo["CM"],
    errors="coerce"
).fillna(0)

DailySewingInfo["Sewing Qty"] = pd.to_numeric(
    DailySewingInfo["Sewing Qty"],
    errors="coerce"
).fillna(0)


# Remove rows without Factory / Line
DailySewingInfo = DailySewingInfo[
    (DailySewingInfo["Factory Name"] != "") &
    (DailySewingInfo["Line No"] != "") &
    (DailySewingInfo["Factory Name"] != "nan") &
    (DailySewingInfo["Line No"] != "nan")
].copy()


# ============================================================
# 6. RECORD-WISE EARNING
# ============================================================
#
# Quantity * CM
#
# Example:
# 1100 * 1.20 = 1320
#
# ============================================================

DailySewingInfo["Record Earning"] = (
    DailySewingInfo["Sewing Qty"] * DailySewingInfo["CM"]
)


# ============================================================
# 7. GROUP FACTORY + LINE
# ============================================================
#
# Important:
#
# First calculate:
#
#     SUM(Quantity * CM)
#
# Then deduct:
#
#     1925
#
# ONLY ONCE per Factory + Line.
#
# ============================================================

grouped = (
    DailySewingInfo.groupby(
        ["Factory Name", "Line No"],
        as_index=False
    )["Record Earning"]
    .sum()
)


# ============================================================
# 8. DEDUCT FIXED LINE COST
# ============================================================

grouped["Earning"] = (
    grouped["Record Earning"]
    - FIXED_LINE_COST
)


# ============================================================
# 9. KEEP ONLY REQUIRED FACTORIES
# ============================================================

grouped = grouped[
    grouped["Factory Name"].isin(FACTORIES)
].copy()


# ============================================================
# 10. PIVOT FACTORY INTO COLUMNS
# ============================================================

pivot = grouped.pivot(
    index="Line No",
    columns="Factory Name",
    values="Earning"
)


# ============================================================
# 11. FORCE FACTORY COLUMN ORDER
# ============================================================

pivot = pivot.reindex(
    columns=FACTORIES
)


# ============================================================
# 12. DYNAMICALLY SORT LINE NUMBERS
# ============================================================

def line_sort_key(value):
    """
    Line-1
    Line-2
    Line-10

    will become:
    1, 2, 10
    instead of:
    1, 10, 2
    """

    value = str(value)

    try:
        return int(
            value.split("-")[-1]
        )
    except ValueError:
        return 999999


pivot = pivot.sort_index(
    key=lambda x: x.map(line_sort_key)
)


# ============================================================
# 13. RENAME FACTORY COLUMNS
# ============================================================

pivot.columns = [
    f"{factory} (Earning)"
    for factory in pivot.columns
]


# ============================================================
# 14. RESET INDEX
# ============================================================

df_summary = pivot.reset_index()


# ============================================================
# 15. ADD "Line wise Earning" BEFORE "Line No"
# ============================================================

df_summary.insert(
    0,
    "Line wise Earning",
    ""
)

# Put text only in first cell
if not df_summary.empty:
    df_summary.loc[
        df_summary.index[0],
        "Line wise Earning"
    ] = LINE_WISE_TEXT


# ============================================================
# 16. FINAL COLUMN ORDER
# ============================================================

line_wise_earning = df_summary[
    [
        "Line wise Earning",
        "Line No",
        "MGNSL (Earning)",
        "MGL (Earning)",
        "MGSL (Earning)",
        "MGSL-DK (Earning)",
    ]
]


# ============================================================
# 17. DISPLAY DATAFRAME
# ============================================================

line_wise_earning

,Line wise Earning,Line No,MGNSL (Earning),MGL (Earning),MGSL (Earning),MGSL-DK (Earning)
0,*Quantity * CM - Fixed Line Cost Per Day USD $...,Line-1,-1029.000,-37.0,-1225.000,-469.00
1,,Line-2,-53.000,-245.0,-1445.000,-568.34
2,,Line-3,-496.952,-485.0,-771.776,-518.88
3,,Line-4,-1177.500,-973.0,-21.000,-245.00
4,,Line-5,-1120.000,-815.0,-21.000,-435.50
5,,Line-6,-1739.400,-725.0,-21.000,NaN
6,,Line-7,-637.000,-425.0,35.000,NaN
7,,Line-8,-765.000,139.0,254.770,NaN
8,,Line-9,-53.000,-187.6,-21.000,NaN
9,,Line-10,-1189.000,51.0,NaN,NaN


In [35]:
reports = [
    ("Report 1", less_production),
    ("Report 2", fabric_delay_df),
    ("Report 3", line_wise_earning)
]

print("Reports:", [name for name, _ in reports])


Reports: ['Report 1', 'Report 2', 'Report 3']


## Data Process End Here

## 3. Build and inspect the local master DataFrame

In [36]:
master_df = build_master_dataframe(reports)
display(master_df)


,0,1,2,3,4,5,6,7
0,Report 1,,,,,,,
1,Less Production,Factory,Line,Buyer,Style,Day Target Qty,Achieve Qty,Percentage
2,,MGL,Line 01,STREETONE,346497/3413355,582,400,68.73
3,,MGL,Line 02,US POLO,AW26-FM-SKT-002,364,120,32.97
4,Yesterday,MGL,Line 06,NEXT,Y63406 - GIRLS PANT,"1,500","1,000",66.67
...,...,...,...,...,...,...,...,...
84,,Line-9,-53.0,-187.6,-21.0,NaN,,
85,,Line-10,-1189.0,51.0,NaN,NaN,,
86,,Line-11,-53.0,NaN,NaN,NaN,,
87,,Line-12,-1925.0,NaN,NaN,NaN,,


## 4. Previous-day target

The system always uses Bangladesh time and writes yesterday into a sheet named `DD_MMM_YYYY`. Example: `2026-09-20` → `20_Sep_2026`.

In [37]:
previous_day = get_previous_report_date()
target_sheet = format_sheet_name(previous_day)

print("Previous report date:", previous_day.isoformat())
print("Target sheet name   :", target_sheet)


Previous report date: 2026-09-23
Target sheet name   : 23_Sep_2026


## 5. Build strict JSON-safe payload

In [38]:
payload = build_payload(
    reports=reports,
    report_date=previous_day,
)

json.dumps(payload, allow_nan=False)

print("Payload version:", payload["version"])
print("Run ID         :", payload["run_id"])
print("Report date    :", payload["report_date"])
print("Report count   :", len(payload["reports"]))
print("Strict JSON    : PASS")


Payload version: 1
Run ID         : d1bed35fe79841539a340e4e4e9d476a
Report date    : 2026-09-23
Report count   : 3
Strict JSON    : PASS


## 6. Upload once

In [39]:
result = upload_to_google_apps_script(payload)

if result.get("success"):
    print("UPLOAD SUCCESS")
    print("Sheet        :", result.get("sheet"))
    print("Reports      :", result.get("report_count"))
    print("Rows written :", result.get("rows_written"))
    print("Run ID       :", result.get("run_id"))
else:
    print("UPLOAD FAILED")
    print("Reason       :", result.get("error"))
    raise RuntimeError(result.get("error", "Unknown upload error"))


UPLOAD SUCCESS
Sheet        : 23_Sep_2026
Reports      : 3
Rows written : 89
Run ID       : d1bed35fe79841539a340e4e4e9d476a


## 7. Local data-type test

Use this once after installation to verify NaN, NaT, NumPy values, booleans, dates, and datetimes serialize correctly.

In [40]:
import datetime as dt
import numpy as np

messy = pd.DataFrame({
    "text": ["ok", None, "00123"],
    "float": [1.5, np.nan, np.inf],
    "int": np.array([1, 2, 3], dtype=np.int64),
    "date": [pd.Timestamp("2026-09-20"), pd.NaT, pd.Timestamp("2026-09-20 14:30:05")],
    "flag": np.array([True, False, True]),
    "pydate": [dt.date(2026, 9, 20), None, dt.datetime(2026, 9, 20, 8, 0, 0)],
})

check_payload = build_payload([("Messy Report", messy)], previous_day)
json.dumps(check_payload, allow_nan=False)
print(json.dumps(check_payload, indent=2, ensure_ascii=False))
print("Strict JSON: PASS")


{
  "version": 1,
  "run_id": "a8fdab6ec69f4ee49a9c3f3dfaa0287c",
  "report_date": "2026-09-23",
  "reports": [
    {
      "name": "Messy Report",
      "headers": [
        "text",
        "float",
        "int",
        "date",
        "flag",
        "pydate"
      ],
      "rows": [
        [
          "ok",
          1.5,
          1,
          "2026-09-20",
          true,
          "2026-09-20"
        ],
        [
          "",
          "",
          2,
          "",
          false,
          ""
        ],
        [
          "00123",
          "",
          3,
          "2026-09-20 14:30:05",
          true,
          "2026-09-20 08:00:00"
        ]
      ]
    }
  ]
}
Strict JSON: PASS
